# Operational Regime-Aware Degradation Simulation
## Mission Planning Support via Conditional Autoregressive Transformer

**IEEE IRAI 2026** | Digvijay Rajoria, Chathurika S. Wickramasinghe Brahmana

---

This notebook demonstrates the **inference pipeline** of our trained model:
- Load pretrained weights
- Simulate regime-conditioned degradation trajectories
- Visualize trajectory divergence under low-stress vs. high-stress regimes
- Show per-sensor divergence heatmap across all 14 sensor channels

> **No training required** — this notebook runs in ~2 minutes on a free Colab T4 GPU.

**Before running:** Mount your Google Drive and ensure model weights are at:
`/content/drive/MyDrive/TheCook/models/`

In [ ]:
# Cell 1 — Setup
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from google.colab import drive

drive.mount('/content/drive')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — Configuration
DATA_PATH   = '/content/drive/MyDrive/TheCook/data/'
MODEL_PATH  = '/content/drive/MyDrive/TheCook/models/'

SENSORS_TO_DROP = [1, 5, 6, 10, 16, 18, 19]
ALL_COLS = (['engine_id', 'cycle'] +
            [f'op{i}' for i in range(1, 4)] +
            [f's{i}' for i in range(1, 22)])
SENSOR_COLS = [f's{i}' for i in range(1, 22) if i not in SENSORS_TO_DROP]
OP_COLS     = ['op1', 'op2', 'op3']
RUL_CAP     = 125
WINDOW      = 30
K           = 10   # trajectory horizon
N_MC        = 30   # MC dropout passes

print(f'Retained sensors ({len(SENSOR_COLS)}): {SENSOR_COLS}')

In [ ]:
# Cell 3 — Data loading and preprocessing
def load_and_preprocess(subset='FD002'):
    train_df = pd.read_csv(f'{DATA_PATH}train_{subset}.txt',
                           sep=r'\s+', header=None, names=ALL_COLS)
    test_df  = pd.read_csv(f'{DATA_PATH}test_{subset}.txt',
                           sep=r'\s+', header=None, names=ALL_COLS)
    rul_df   = pd.read_csv(f'{DATA_PATH}RUL_{subset}.txt',
                           header=None, names=['RUL'])

    train_df['RUL'] = train_df.groupby('engine_id')['cycle'].transform(
        lambda x: x.max() - x)
    train_df['RUL'] = train_df['RUL'].clip(upper=RUL_CAP)

    scaler    = MinMaxScaler()
    op_scaler = MinMaxScaler()
    train_df[SENSOR_COLS] = scaler.fit_transform(train_df[SENSOR_COLS])
    test_df[SENSOR_COLS]  = scaler.transform(test_df[SENSOR_COLS])
    train_df[OP_COLS] = op_scaler.fit_transform(train_df[OP_COLS])
    test_df[OP_COLS]  = op_scaler.transform(test_df[OP_COLS])

    # Test windows — last WINDOW cycles per engine
    X_te, OPS_te, Y_te = [], [], []
    for eid, grp in test_df.groupby('engine_id'):
        sensors  = grp[SENSOR_COLS].values
        ops      = grp[OP_COLS].values
        true_rul = min(float(rul_df.loc[eid - 1, 'RUL']), RUL_CAP)
        if len(grp) < WINDOW:
            pad = WINDOW - len(grp)
            sensors = np.vstack([np.zeros((pad, sensors.shape[1])), sensors])
            ops     = np.vstack([np.zeros((pad, ops.shape[1])), ops])
        X_te.append(sensors[-WINDOW:])
        OPS_te.append(ops[-WINDOW:])
        Y_te.append(true_rul)

    # Training ops for K-means clustering
    OPS_tr_flat = train_df[OP_COLS].values

    print(f'{subset} → Test engines: {len(X_te)}')
    return (np.array(X_te, dtype=np.float32),
            np.array(OPS_te, dtype=np.float32),
            np.array(Y_te, dtype=np.float32),
            OPS_tr_flat, scaler, op_scaler)

X_te2, OPS_te2, Y_te2, OPS_tr_flat, scaler2, op_scaler2 = load_and_preprocess('FD002')
print('\nData loaded successfully.')

In [ ]:
# Cell 4 — Model architecture definition
class RegimeAwareARModel(nn.Module):
    """
    Conditional Autoregressive Transformer for Regime-Aware Degradation Simulation.
    Architecture:
      - 2-layer Transformer encoder with cross-attention regime fusion
      - GRU-based autoregressive trajectory decoder
      - MC Dropout uncertainty estimation (p=0.1)
    """
    def __init__(self, n_feat=14, n_op=3, d_model=64,
                 n_heads=4, n_layers=2, K=10, dropout_p=0.1):
        super().__init__()
        self.K = K
        self.d_model = d_model
        self.n_feat  = n_feat
        self.dropout_p = dropout_p

        # Sensor encoder
        self.sensor_proj = nn.Linear(n_feat, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=128, dropout=dropout_p,
            batch_first=True)
        self.sensor_enc = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        # Regime encoder
        self.regime_proj = nn.Linear(n_op, d_model)

        # Cross-attention fusion
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads,
            dropout=dropout_p, batch_first=True)
        self.cross_norm = nn.LayerNorm(d_model)

        # RUL head
        self.rul_head = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid())

        # Autoregressive decoder
        self.decoder_proj = nn.Linear(n_feat + n_op, d_model)
        self.gru_cell     = nn.GRUCell(d_model, d_model)
        self.dec_dropout  = nn.Dropout(dropout_p)
        self.dec_cross_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads,
            dropout=dropout_p, batch_first=True)
        self.dec_cross_norm = nn.LayerNorm(d_model)
        self.output_proj  = nn.Sequential(
            nn.Linear(d_model, n_feat), nn.Sigmoid())

    def encode(self, X, OPS):
        s = self.sensor_proj(X)
        s = self.sensor_enc(s)
        r = self.regime_proj(OPS)
        fused, _ = self.cross_attn(query=s, key=r, value=r)
        F = self.cross_norm(fused + s)
        h = F.mean(dim=1)
        rul = self.rul_head(h)
        return rul, F, h

    def decode(self, F, h, future_ops):
        B, K_steps, _ = future_ops.shape
        s_prev = torch.zeros(B, self.n_feat, device=F.device)
        h_gru  = h.clone()
        preds  = []
        for k in range(K_steps):
            inp = torch.cat([s_prev, future_ops[:, k, :]], dim=-1)
            a   = self.decoder_proj(inp)
            h_gru = self.gru_cell(a, h_gru)
            h_gru = self.dec_dropout(h_gru)
            q = h_gru.unsqueeze(1)
            z, _ = self.dec_cross_attn(query=q, key=F, value=F)
            z = self.dec_cross_norm(z.squeeze(1) + h_gru)
            s_next = self.output_proj(z)
            preds.append(s_next.unsqueeze(1))
            s_prev = s_next
        return torch.cat(preds, dim=1)

    def forward(self, X, OPS, future_ops=None):
        rul, F, h = self.encode(X, OPS)
        if future_ops is None:
            future_ops = OPS[:, -self.K:, :]
        traj = self.decode(F, h, future_ops)
        return rul, traj

print('Model class defined.')

In [ ]:
# Cell 5 — Load pretrained weights
model_ar = RegimeAwareARModel(K=K).to(device)
model_ar.load_state_dict(
    torch.load(MODEL_PATH + 'model_ar_fd002_dropout.pth',
               map_location=device))
model_ar.eval()
print('✅ Pretrained model loaded successfully.')

In [ ]:
# Cell 6 — K-means regime clustering (reproduce low/high stress centroids)
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
kmeans.fit(OPS_tr_flat)
centroids = kmeans.cluster_centers_
order     = np.argsort(centroids.mean(axis=1))
c_low  = torch.tensor(centroids[order[0]],  dtype=torch.float32)  # lowest stress
c_high = torch.tensor(centroids[order[-1]], dtype=torch.float32)  # highest stress

print(f'Low-stress centroid:  {c_low.numpy().round(3)}')
print(f'High-stress centroid: {c_high.numpy().round(3)}')

In [ ]:
# Cell 7 — MC Dropout trajectory inference function
def mc_trajectory(model, X, OPS, future_ops, N=30, K=10):
    """
    Run N stochastic forward passes with dropout enabled.
    Returns mean trajectory and 1-sigma uncertainty bands.
    """
    model.train()  # keep dropout active
    all_trajs = []
    with torch.no_grad():
        for _ in range(N):
            _, traj = model(X, OPS, future_ops)
            all_trajs.append(traj.cpu().numpy())
    model.eval()
    stack = np.stack(all_trajs, axis=0)       # (N, B, K, 14)
    mean  = stack.mean(axis=0)                # (B, K, 14)
    sigma = stack.std(axis=0)                 # (B, K, 14)
    return mean, sigma

print('MC trajectory function ready.')

In [ ]:
# Cell 8 — Generate trajectories for publication figure (3 engines)
# Engines chosen to span the full RUL range (RUL ~18, ~125, ~30)
# Find engines closest to these RUL values
target_ruls = [18, 125, 30]
engine_idxs = [int(np.argmin(np.abs(Y_te2 - t))) for t in target_ruls]
print(f'Selected engine indices: {engine_idxs}')
print(f'Their true RULs: {Y_te2[engine_idxs].round(1)}')

results = {}
for eng_idx in engine_idxs:
    X_e   = torch.tensor(X_te2[eng_idx]).unsqueeze(0).to(device)
    OPS_e = torch.tensor(OPS_te2[eng_idx]).unsqueeze(0).to(device)

    ops_low  = c_low.expand(1, K, 3).to(device)
    ops_high = c_high.expand(1, K, 3).to(device)

    mean_low,  sigma_low  = mc_trajectory(model_ar, X_e, OPS_e, ops_low,  N=N_MC, K=K)
    mean_high, sigma_high = mc_trajectory(model_ar, X_e, OPS_e, ops_high, N=N_MC, K=K)

    results[eng_idx] = {
        'rul': Y_te2[eng_idx],
        'mean_low': mean_low[0],   # (K, 14)
        'mean_high': mean_high[0],
        'sigma_low': sigma_low[0],
        'sigma_high': sigma_high[0],
    }

print('\n✅ Trajectories generated for all engines.')

In [ ]:
# Cell 9 — Publication figure: s15 (bypass ratio) trajectories
S15_IDX = SENSOR_COLS.index('s15')
steps   = np.arange(0, K + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
fig.suptitle(
    'Regime-Conditioned Degradation Trajectories: Same Observed State, Divergent Future Evolution\n'
    f'(N={N_MC} MC dropout passes; shaded regions: ±1σ epistemic uncertainty)',
    fontsize=11)

for ax, eng_idx in zip(axes, engine_idxs):
    r = results[eng_idx]
    rul_val = r['rul']

    # s15 values: start at observed, then predicted
    obs_val    = X_te2[eng_idx, -1, S15_IDX]   # last observed
    traj_low   = np.concatenate([[obs_val], r['mean_low'][:, S15_IDX]])
    traj_high  = np.concatenate([[obs_val], r['mean_high'][:, S15_IDX]])
    sigma_low  = np.concatenate([[0], r['sigma_low'][:, S15_IDX]])
    sigma_high = np.concatenate([[0], r['sigma_high'][:, S15_IDX]])

    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
    ax.plot(steps, traj_low,  'b-o', linewidth=1.5, markersize=4, label='Low-stress')
    ax.plot(steps, traj_high, 'r-s', linewidth=1.5, markersize=4, label='High-stress')
    ax.fill_between(steps, traj_low - sigma_low,   traj_low + sigma_low,
                    alpha=0.2, color='blue')
    ax.fill_between(steps, traj_high - sigma_high, traj_high + sigma_high,
                    alpha=0.2, color='red')

    ax.set_title(f'Engine {eng_idx+1}  (True RUL: {int(rul_val)} cycles)')
    ax.set_xlabel('Future Cycle Steps')
    ax.set_ylabel('Sensor s15 (Bypass Ratio)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('trajectory_figure_s15.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved: trajectory_figure_s15.png')

In [ ]:
# Cell 10 — Per-sensor divergence: all 14 sensors × 6 conditions
N_ENGINES = 10

# 6-condition K-means clustering
centroids_sorted = centroids[order]   # already sorted by severity
regime_base = torch.tensor(centroids_sorted[0], dtype=torch.float32)  # lowest

divergence_matrix = np.zeros((len(SENSOR_COLS), 6))
engine_idxs_sweep = list(range(min(N_ENGINES, len(X_te2))))

for cond_idx in range(6):
    regime_target = torch.tensor(centroids_sorted[cond_idx], dtype=torch.float32)
    per_engine_div = []
    for eng_idx in engine_idxs_sweep:
        X_e   = torch.tensor(X_te2[eng_idx]).unsqueeze(0).to(device)
        OPS_e = torch.tensor(OPS_te2[eng_idx]).unsqueeze(0).to(device)
        ops_b = regime_base.expand(1, K, 3).to(device)
        ops_t = regime_target.expand(1, K, 3).to(device)

        mean_b, _ = mc_trajectory(model_ar, X_e, OPS_e, ops_b, N=N_MC, K=K)
        mean_t, _ = mc_trajectory(model_ar, X_e, OPS_e, ops_t, N=N_MC, K=K)

        div = np.abs(mean_t - mean_b).mean(axis=1).squeeze(0)   # (14,)
        per_engine_div.append(div)

    divergence_matrix[:, cond_idx] = np.mean(per_engine_div, axis=0)

print(f'✅ Divergence matrix shape: {divergence_matrix.shape}  (14 sensors × 6 conditions)')

# Summary stats
extreme_div = divergence_matrix[:, 5]   # extreme high-stress vs. base
mean_div    = divergence_matrix.mean(axis=1)
print(f'\n14-sensor mean L1 divergence (extreme conditions): {extreme_div.mean():.3f}')
print(f'Range: {extreme_div.min():.3f} – {extreme_div.max():.3f}')

In [ ]:
# Cell 11 — Sensor divergence heatmap
fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(divergence_matrix, aspect='auto', cmap='viridis')
ax.set_xticks(range(6))
ax.set_xticklabels([f'Cond {i}\n(severity rank {i})' for i in range(6)], fontsize=8)
ax.set_yticks(range(len(SENSOR_COLS)))
ax.set_yticklabels(SENSOR_COLS, fontsize=9)
ax.set_xlabel('Operating Condition (0 = lowest stress)', fontsize=10)
ax.set_title(
    f'Mean L1 Trajectory Divergence vs. Lowest-Stress Baseline\n'
    f'(per sensor / condition, {N_ENGINES} engines, N={N_MC} MC passes)', fontsize=10)
plt.colorbar(im, ax=ax, label='Mean L1 divergence')
plt.tight_layout()
plt.savefig('sensor_divergence_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved: sensor_divergence_heatmap.png')

# Ranked sensor table
ranked = np.argsort(extreme_div)[::-1]
print('\nSensor divergence ranking (extreme low vs. high stress):')
for rank, idx in enumerate(ranked, 1):
    marker = '  ← used in paper (Fig. 2)' if SENSOR_COLS[idx] == 's15' else ''
    print(f'  {rank:2d}. {SENSOR_COLS[idx]}: {extreme_div[idx]:.4f}{marker}')

In [ ]:
# Cell 12 — RUL prediction accuracy check
from torch.utils.data import Dataset, DataLoader

class CMAPSSDataset(Dataset):
    def __init__(self, X, OPS, Y):
        self.X   = torch.tensor(X)
        self.OPS = torch.tensor(OPS)
        self.Y   = torch.tensor(Y).unsqueeze(1)
    def __len__(self): return len(self.Y)
    def __getitem__(self, i): return self.X[i], self.OPS[i], self.Y[i]

te_ds     = CMAPSSDataset(X_te2, OPS_te2, Y_te2)
te_loader = DataLoader(te_ds, batch_size=256)

model_ar.eval()
preds, trues = [], []
with torch.no_grad():
    for X, OPS, Y in te_loader:
        X, OPS = X.to(device), OPS.to(device)
        rul, _ = model_ar(X, OPS)
        preds.append(rul.cpu().numpy() * RUL_CAP)
        trues.append(Y.numpy())

preds = np.concatenate(preds)
trues = np.concatenate(trues)
rmse  = float(np.sqrt(np.mean((preds - trues)**2)))

d = preds.squeeze() - trues.squeeze()
phm = float(np.sum(np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1)))

print(f'FD002 RMSE:      {rmse:.2f}  (paper: 17.34)')
print(f'FD002 PHM Score: {phm:.0f}   (paper: 1476)')
print('\n(Small variation expected — no fixed random seed was used during training)')

---

## Summary of Results

| Finding | Value |
|---|---|
| FD002 RMSE (base model) | ~17.34 |
| DAST benchmark (Zhang et al. 2022) | 18.19 |
| Mean L1 trajectory divergence (14 sensors) | 0.586 |
| Most regime-sensitive sensor | s12 (0.91) |
| s15 (bypass ratio, used in paper figure) | 0.32 |

**Key insight:** Regime conditioning does not improve RUL accuracy (sensor readings already encode regime state implicitly) — but it is indispensable for counterfactual trajectory simulation under user-specified future regimes.

---

## Citation

```bibtex
@inproceedings{rajoria2026regime,
  title     = {Operational Regime-Aware Degradation Simulation for Mission Planning Support},
  author    = {Rajoria, Digvijay and Wickramasinghe Brahmana, Chathurika S.},
  booktitle = {Proc. IEEE International Conference on Responsible Artificial Intelligence (IRAI)},
  year      = {2026},
  address   = {Melbourne, Australia}
}
```

*Developed as part of the IEEE IES Generative AI Challenge 2026.*